# Label-free small-molecule representation inspector

**Mode:** editable student exercise. This notebook uses seven authored teaching examples embedded below. It does not download data, load activity labels, fit a model, make a split, or report predictive performance.

For Colab, download this `.ipynb` file and choose **File > Upload notebook**. Local Jupyter is an alternative. The setup cell requires Python 3.11 or newer and uses RDKit 2025.9.6. It installs that exact package only when missing or mismatched; if it changes RDKit, restart the runtime before rerunning the notebook. No GPU is needed. This notebook has not been verified in Colab.

It creates `representation-report.json` in a fresh directory beneath the notebook's current working directory and retains it for download. It never reads or changes author results, caches, or local project files. Delete the generated directory yourself when it is no longer needed. Resetting the runtime clears variables; rerun cells in order.

In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_PYTHON = (3, 11)
REQUIRED_RDKIT = "2025.9.6"


def normalized_version(value):
    return tuple(int(part) for part in value.split("."))

if sys.version_info < REQUIRED_PYTHON:
    raise RuntimeError(
        f"Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]} or newer is required; "
        f"this runtime is {sys.version.split()[0]}."
    )

try:
    installed_rdkit = importlib.metadata.version("rdkit")
except importlib.metadata.PackageNotFoundError:
    installed_rdkit = None

loaded_rdkit = sys.modules.get("rdkit")
if loaded_rdkit is not None and normalized_version(
    getattr(loaded_rdkit, "__version__", "0")
) != normalized_version(REQUIRED_RDKIT):
    raise RuntimeError(
        "A different RDKit version is already loaded. Restart the runtime, then rerun "
        "this setup cell so it can install the pinned version safely."
    )

if installed_rdkit is None or normalized_version(installed_rdkit) != normalized_version(
    REQUIRED_RDKIT
):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", f"rdkit=={REQUIRED_RDKIT}"]
    )
    raise RuntimeError(
        "RDKit was installed or changed. Restart the runtime, then rerun this notebook "
        "from the first cell."
    )

import rdkit

if normalized_version(rdkit.__version__) != normalized_version(REQUIRED_RDKIT):
    raise RuntimeError(
        f"Loaded RDKit {rdkit.__version__}, expected {REQUIRED_RDKIT}. Restart the runtime."
    )

print(f"Python {sys.version.split()[0]}; RDKit {rdkit.__version__}")

## Question and computational objects

A SMILES string is an input notation, not an experimental measurement. RDKit parses that notation into a molecular graph, then derives canonical isomeric and non-isomeric SMILES, an InChIKey, simple 2D descriptors, a Murcko scaffold, and two hashed Morgan fingerprint summaries. The code reports a binary fingerprint's occupied bit count and a count fingerprint's nonzero-feature count. Neither number is a biological or physicochemical target.

The next code cell embeds the repository's tested inspection dataclasses and functions verbatim. Keeping them inside the notebook makes the uploaded file self-contained while preserving the canonical radius 2, 2,048-bit, chirality-aware Morgan configuration.

In [ ]:
from __future__ import annotations

import json
from collections.abc import Sequence
from dataclasses import asdict, dataclass

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold


class RepresentationInputError(ValueError):
    """Raised when a label-free representation input violates the lab contract."""


@dataclass(frozen=True)
class MoleculeInput:
    source_id: str
    smiles: str
    teaching_point: str


@dataclass(frozen=True)
class RepresentationConfig:
    morgan_radius: int = 2
    morgan_bits: int = 2048
    include_chirality: bool = True

    def __post_init__(self) -> None:
        if self.morgan_radius < 0:
            raise RepresentationInputError("Morgan radius must be non-negative")
        if self.morgan_bits <= 0:
            raise RepresentationInputError("Morgan bit-vector size must be positive")


@dataclass(frozen=True)
class DescriptorSnapshot:
    molecular_weight: float
    wildman_crippen_logp: float
    topological_polar_surface_area: float
    hydrogen_bond_donors: int
    hydrogen_bond_acceptors: int
    rotatable_bonds: int


@dataclass(frozen=True)
class MoleculeInspection:
    source_id: str
    input_smiles: str
    teaching_point: str
    canonical_isomeric_smiles: str
    canonical_nonisomeric_smiles: str
    inchi_key: str
    molecular_formula: str
    atom_count: int
    heavy_atom_count: int
    bond_count: int
    component_count: int
    formal_charge: int
    specified_stereocenters: int
    unassigned_stereocenters: int
    murcko_scaffold_smiles: str | None
    descriptors: DescriptorSnapshot
    morgan_on_bits: int
    morgan_nonzero_count_features: int


@dataclass(frozen=True)
class RepresentationContract:
    rdkit_version: str
    canonical_smiles_toolkit: str
    descriptor_names: tuple[str, ...]
    morgan_radius: int
    morgan_bits: int
    include_chirality: bool
    fingerprint_forms: tuple[str, ...]


@dataclass(frozen=True)
class RepresentationReport:
    scope: str
    contract: RepresentationContract
    molecules: tuple[MoleculeInspection, ...]


def inspect_molecules(
    inputs: Sequence[MoleculeInput],
    *,
    config: RepresentationConfig | None = None,
) -> tuple[MoleculeInspection, ...]:
    """Inspect molecular encodings without loading labels or fitting a model."""

    resolved_config = config or RepresentationConfig()
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=resolved_config.morgan_radius,
        fpSize=resolved_config.morgan_bits,
        includeChirality=resolved_config.include_chirality,
    )
    seen_ids: set[str] = set()
    inspections: list[MoleculeInspection] = []

    for record in inputs:
        if not record.source_id.strip():
            raise RepresentationInputError("source ID must not be empty")
        if record.source_id in seen_ids:
            raise RepresentationInputError(f"duplicate source ID: {record.source_id}")
        seen_ids.add(record.source_id)

        molecule = Chem.MolFromSmiles(record.smiles) if record.smiles.strip() else None
        if molecule is None or molecule.GetNumAtoms() == 0:
            raise RepresentationInputError(
                f"invalid or zero-atom SMILES for source ID {record.source_id}"
            )

        stereocenters = Chem.FindMolChiralCenters(  # type: ignore[no-untyped-call]
            molecule,
            force=True,
            includeUnassigned=True,
            includeCIP=True,
            useLegacyImplementation=False,
        )
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(  # type: ignore[no-untyped-call]
            mol=molecule
        )
        bit_fingerprint = generator.GetFingerprint(molecule)
        count_fingerprint = generator.GetCountFingerprint(molecule)
        inspections.append(
            MoleculeInspection(
                source_id=record.source_id,
                input_smiles=record.smiles,
                teaching_point=record.teaching_point,
                canonical_isomeric_smiles=Chem.MolToSmiles(
                    molecule, isomericSmiles=True
                ),
                canonical_nonisomeric_smiles=Chem.MolToSmiles(
                    molecule, isomericSmiles=False
                ),
                inchi_key=Chem.MolToInchiKey(molecule),  # type: ignore[no-untyped-call]
                molecular_formula=rdMolDescriptors.CalcMolFormula(molecule),
                atom_count=molecule.GetNumAtoms(),
                heavy_atom_count=molecule.GetNumHeavyAtoms(),
                bond_count=molecule.GetNumBonds(),
                component_count=len(Chem.GetMolFrags(molecule)),
                formal_charge=sum(
                    atom.GetFormalCharge()
                    for atom in molecule.GetAtoms()  # type: ignore[no-untyped-call]
                ),
                specified_stereocenters=sum(
                    assignment != "?" for _, assignment in stereocenters
                ),
                unassigned_stereocenters=sum(
                    assignment == "?" for _, assignment in stereocenters
                ),
                murcko_scaffold_smiles=scaffold or None,
                descriptors=DescriptorSnapshot(
                    molecular_weight=float(
                        Descriptors.MolWt(molecule)  # type: ignore[attr-defined]
                    ),
                    wildman_crippen_logp=float(
                        Descriptors.MolLogP(molecule)  # type: ignore[attr-defined]
                    ),
                    topological_polar_surface_area=float(
                        rdMolDescriptors.CalcTPSA(molecule)
                    ),
                    hydrogen_bond_donors=int(rdMolDescriptors.CalcNumHBD(molecule)),
                    hydrogen_bond_acceptors=int(rdMolDescriptors.CalcNumHBA(molecule)),
                    rotatable_bonds=int(
                        rdMolDescriptors.CalcNumRotatableBonds(molecule)
                    ),
                ),
                morgan_on_bits=len(bit_fingerprint.GetOnBits()),
                morgan_nonzero_count_features=len(
                    count_fingerprint.GetNonzeroElements()
                ),
            )
        )

    return tuple(inspections)


def build_report(
    inputs: Sequence[MoleculeInput],
    *,
    config: RepresentationConfig | None = None,
) -> RepresentationReport:
    resolved_config = config or RepresentationConfig()
    return RepresentationReport(
        scope="label-free representation inspection",
        contract=RepresentationContract(
            rdkit_version=rdBase.rdkitVersion,
            canonical_smiles_toolkit="RDKit",
            descriptor_names=(
                "molecular_weight",
                "wildman_crippen_logp",
                "topological_polar_surface_area",
                "hydrogen_bond_donors",
                "hydrogen_bond_acceptors",
                "rotatable_bonds",
            ),
            morgan_radius=resolved_config.morgan_radius,
            morgan_bits=resolved_config.morgan_bits,
            include_chirality=resolved_config.include_chirality,
            fingerprint_forms=("hashed_bit_vector", "hashed_count_vector"),
        ),
        molecules=inspect_molecules(inputs, config=resolved_config),
    )


def render_report(report: RepresentationReport) -> str:
    return json.dumps(asdict(report), indent=2, sort_keys=True) + "\n"

## Seven authored examples

The pair of pyridine strings should canonicalize to the same RDKit identity. The two defined alanines should keep distinct isomeric representations while sharing a non-isomeric one. The unspecified alanine is not assigned to either enantiomer. Sodium acetate remains a two-component record; the notebook neither chooses a parent fragment nor neutralizes it.

In [ ]:
EXAMPLES = (
    MoleculeInput(
        "pyridine-kekule",
        "C1=CC=CN=C1",
        "Equivalent graph written with explicit alternating bonds",
    ),
    MoleculeInput(
        "pyridine-aromatic",
        "n1ccccc1",
        "Equivalent graph written with aromatic atoms",
    ),
    MoleculeInput(
        "alanine-one",
        "N[C@@H](C)C(=O)O",
        "One defined tetrahedral stereoisomer",
    ),
    MoleculeInput(
        "alanine-two",
        "N[C@H](C)C(=O)O",
        "The opposite defined tetrahedral stereoisomer",
    ),
    MoleculeInput(
        "alanine-unspecified",
        "NC(C)C(=O)O",
        "Connectivity retained but tetrahedral stereochemistry unspecified",
    ),
    MoleculeInput("acetic-acid", "CC(=O)O", "Neutral protonation state"),
    MoleculeInput(
        "sodium-acetate",
        "CC(=O)[O-].[Na+]",
        "Disconnected salt with two components",
    ),
)
CONFIG = RepresentationConfig(
    morgan_radius=2,
    morgan_bits=2048,
    include_chirality=True,
)
report = build_report(EXAMPLES, config=CONFIG)
report_payload = json.loads(render_report(report))

assert len(report.molecules) == len(EXAMPLES)
assert report.contract.morgan_radius == 2
assert report.contract.morgan_bits == 2048
assert report.contract.include_chirality is True
print(f"Inspected {len(report.molecules)} label-free examples.")

## Inspect, compare, and draw

The tables compare graph-derived fields. The SVG drawings make the input graph visible; they do not establish a solution conformation, binding pose, affinity, efficacy, safety, or synthesizability. Canonicalization also does not decide protonation, tautomer state, or assay identity.

In [ ]:
from html import escape

from IPython.display import HTML, SVG, display
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D


def html_table(headers, rows):
    header_html = "".join(f"<th>{escape(str(header))}</th>" for header in headers)
    row_html = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    return (
        "<table><thead><tr>"
        + header_html
        + "</tr></thead><tbody>"
        + row_html
        + "</tbody></table>"
    )


def molecule_svg(smiles, legend):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise RepresentationInputError(f"could not draw invalid SMILES: {smiles}")
    drawer = rdMolDraw2D.MolDraw2DSVG(300, 220)
    rdMolDraw2D.PrepareAndDrawMolecule(drawer, molecule, legend=legend)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


by_id = {item.source_id: item for item in report.molecules}
display(
    HTML(
        html_table(
            ("example", "input", "canonical isomeric SMILES", "formula", "components"),
            [
                (
                    item.source_id,
                    item.input_smiles,
                    item.canonical_isomeric_smiles,
                    item.molecular_formula,
                    item.component_count,
                )
                for item in report.molecules
            ],
        )
    )
)

display(
    HTML(
        html_table(
            ("pyridine input", "canonical isomeric SMILES", "InChIKey"),
            [
                (item.input_smiles, item.canonical_isomeric_smiles, item.inchi_key)
                for item in (
                    by_id["pyridine-kekule"],
                    by_id["pyridine-aromatic"],
                )
            ],
        )
    )
)
display(
    HTML(
        html_table(
            (
                "alanine example",
                "isomeric SMILES",
                "non-isomeric SMILES",
                "specified / unassigned stereocenters",
            ),
            [
                (
                    item.source_id,
                    item.canonical_isomeric_smiles,
                    item.canonical_nonisomeric_smiles,
                    f"{item.specified_stereocenters} / {item.unassigned_stereocenters}",
                )
                for item in (
                    by_id["alanine-one"],
                    by_id["alanine-two"],
                    by_id["alanine-unspecified"],
                )
            ],
        )
    )
)
sodium_acetate = by_id["sodium-acetate"]
display(
    HTML(
        html_table(
            ("salt", "canonical isomeric SMILES", "components", "formal charge"),
            [
                (
                    sodium_acetate.source_id,
                    sodium_acetate.canonical_isomeric_smiles,
                    sodium_acetate.component_count,
                    sodium_acetate.formal_charge,
                )
            ],
        )
    )
)

for item in report.molecules:
    display(HTML(f"<h4>{escape(item.source_id)}</h4>"))
    display(SVG(molecule_svg(item.input_smiles, item.source_id)))

## Fresh output

The output cell creates a fresh retained directory beneath its current working directory, exclusively creates `representation-report.json`, prints its path, and displays a notebook-relative download link. The same directory is visible in the notebook file browser. Download the report before ending a temporary runtime. Delete that generated directory yourself when it is no longer needed; do not point this exercise at a frozen author-results directory.

In [ ]:
import tempfile
from pathlib import Path

from IPython.display import FileLink, display

output_directory = Path(
    tempfile.mkdtemp(prefix="aidd-representation-inspector-", dir=Path.cwd())
)
report_path = output_directory / "representation-report.json"
with report_path.open("x", encoding="utf-8") as report_file:
    report_file.write(render_report(report))
report_json = report_path.read_text(encoding="utf-8")

print(f"REPORT_PATH={report_path}")
print("Wrote a fresh JSON report and retained it for download.")
display(FileLink(report_path.relative_to(Path.cwd()).as_posix(), result_html_prefix="Download: "))

print(
    "REPORT_JSON="
    + json.dumps(
        json.loads(report_json),
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    )
)

## Exercises and answers

1. Before looking at the tables, predict whether the two pyridine inputs will have the same canonical isomeric SMILES and InChIKey. **Answer:** yes in this RDKit version; they encode the same parsed graph.
2. Which alanine field loses the distinction between the two defined enantiomers? **Answer:** canonical non-isomeric SMILES; the isomeric strings and InChIKeys remain different.
3. Does a net-zero sodium acetate record become one neutral molecule here? **Answer:** no. It remains two components, even though the summed formal charge is zero.
4. Add one valid SMILES with a new source ID to `EXAMPLES`. Predict which reported fields can change. Then replace it with `not-a-smiles`. **Answer:** a valid molecule adds one additional inspected record; the malformed input raises `RepresentationInputError` rather than being silently excluded.

This is a representation exercise, not a QSAR, endpoint, training, or evaluation protocol.